In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
!{sys.executable} -m pip install matplotlib mplsoccer numpy scipy pandas shapely

In [ ]:
# Cell 1: Environment setup and root path resolution
# Automatically detect project root regardless of where notebook is executed
current_dir = Path.cwd().resolve()
project_root = current_dir
while not (project_root / "src").exists() and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import custom DLOI-Pose modules from src/
from src.features.lane_occlusion import DefensiveLaneOcclusion
from src.features.pose_orientation import fold_to_tracking_frame, shoulder_orientation_deg
from src.visualization.pitch_viz import plot_defensive_lanes

print(f"Environment successfully configured. Project root: {project_root}")

In [ ]:
# ==============================================================================
# Cell 2: Load Full Phase 406 Stream & Filter Out Substituted "Ghost" Players
# ==============================================================================
from src.data.data_loader import (
    stream_jsonl,
    identify_active_players,
    build_team_lookup,
    compute_frame_velocities,
)

possible_paths = [
    project_root / "sample_1925299_phase406.jsonl",
    project_root / "data" / "bodypose" / "sample_1925299_phase406.jsonl",
    Path("sample_1925299_phase406.jsonl"),
]
sample_path = next((p for p in possible_paths if p.exists() and p.is_file()), None)
if sample_path is None:
    raise FileNotFoundError("Could not find 'sample_1925299_phase406.jsonl'.")

frames = list(stream_jsonl(sample_path))
active_pids = identify_active_players(frames)
team_lookup = build_team_lookup(project_root)

print(f"Loaded {len(frames)} frames (25fps) from Phase 406.")
print(f"Total player records per frame: {len(frames[0]['player_data'])}")
print(f"Active on-pitch players (filtering out 10 substituted ghosts): {len(active_pids)}")

In [ ]:
# ==============================================================================
# Cell 3: Extract All 22 Active Players, Velocities & Names
# ==============================================================================
eval_idx = 0  # Frame 124487 (or set eval_idx = 60 for Frame 124547 at pass release)
frame_record = frames[eval_idx]
velocities = compute_frame_velocities(frames, frame_idx=eval_idx, window=5, fps=25.0)

ball_xy = np.array([frame_record["ball_data"]["x"], frame_record["ball_data"]["y"]], dtype=np.float64)
carrier_id = frame_record["possession"]["player_id"] or 287934
carrier = [p for p in frame_record["player_data"] if p["player_id"] == carrier_id][0]
carrier_xy = np.array([carrier["x"], carrier["y"]], dtype=np.float64)

gk_id = 560985  # Cameron Cook (Perth Glory GK)
gk_list = [p for p in frame_record["player_data"] if p["player_id"] == gk_id]
goalkeeper_xy = np.array([gk_list[0]["x"], gk_list[0]["y"]]) if gk_list else None

defenders, defenders_vel_list, defenders_names, defenders_detected = [], [], [], []
teammates, teammates_names = [], []

for p in frame_record["player_data"]:
    pid = p["player_id"]
    if pid not in active_pids or pid in (carrier_id, gk_id, 50999):
        continue
    meta = team_lookup.get(pid, {})
    t_name = meta.get("team_name", "")
    s_name = meta.get("short_name", f"#{pid}")

    if "Perth Glory" in t_name:
        defenders.append(p)
        defenders_vel_list.append(velocities.get(pid, np.zeros(2)))
        defenders_names.append(s_name)
        defenders_detected.append(bool(p.get("is_detected", False)))
    elif "Brisbane Roar" in t_name:
        teammates.append(p)
        teammates_names.append(s_name)

defenders_xy = np.array([[d["x"], d["y"]] for d in defenders], dtype=np.float64)
defenders_vel = np.array(defenders_vel_list, dtype=np.float64)
teammates_xy = np.array([[t["x"], t["y"]] for t in teammates], dtype=np.float64)

print(f"Loaded {len(defenders)} Perth Glory Outfield Defenders and {len(teammates)} Brisbane Roar Teammates.")

In [ ]:
# ==============================================================================
# Cell 4: Evaluate Tactical Passing Options (Ground vs 3D Aerial Lob)
# ==============================================================================
engine = DefensiveLaneOcclusion()

# Definícia taktických možností: (cieľové súradnice [x, y], príznak is_aerial)
passing_options = {
    "Wide Wing Space (Brazete Run)": (np.array([22.0, 24.5]), False),
    "Right Flank to Feet (J. Brazete)": (np.array([18.62, 16.03]), False),
    "Midfield Pocket (J. O'Shea)": (np.array([16.89, -2.18]), False),
    "Left Half-Space (K. Jelacic)": (np.array([21.21, -11.89]), False),
    "Aerial Channel Lob (Jelacic Run)": (np.array([27.5, -14.0]), True),
    "Central Line Break (Pennington)": (np.array([25.0, 0.0]), False),
    "Backpass Safety (L. Herrington)": (np.array([-5.83, -2.71]), False),
}

results = []
targets_for_plot = {}

for name, (target_xy, is_air) in passing_options.items():
    base_score = engine.compute_frame_occlusion(
        carrier_xy, target_xy, defenders_xy, defenders_vel=defenders_vel,
        defenders_meta=defenders, is_aerial=is_air, use_pose=False
    )
    pose_score = engine.compute_frame_occlusion(
        carrier_xy, target_xy, defenders_xy, defenders_vel=defenders_vel,
        defenders_meta=defenders, is_aerial=is_air, use_pose=True
    )
    delta_pct = ((pose_score - base_score) / (base_score + 1e-6)) * 100
    
    results.append({
        "Passing Option": name,
        "Type": "3D Aerial Lob" if is_air else "Ground Pass",
        "Target (x, y)": f"({target_xy[0]:.1f}, {target_xy[1]:.1f})",
        "Baseline 2D": round(base_score, 2),
        "DLOI-Pose (3D)": round(pose_score, 2),
        "Pose Impact": f"{delta_pct:+.1f}%",
    })
    targets_for_plot[name] = (target_xy, pose_score)

df_eval = pd.DataFrame(results)
display(df_eval)

In [ ]:
# ==============================================================================
# Cell 5: Render Publication-Grade Pitch Graphic with Cover Shadows & Badges
# ==============================================================================
assets_dir = project_root / "assets"
assets_dir.mkdir(exist_ok=True)
output_figure_path = str(assets_dir / "figure1_dloi_pose.png")

defenders_headings = [engine.extract_pose_heading(d) for d in defenders]

fig = plot_defensive_lanes(
    carrier_xy=carrier_xy,
    ball_xy=ball_xy,
    defenders_xy=defenders_xy,
    defenders_headings=defenders_headings,
    targets=targets_for_plot,
    teammates_xy=teammates_xy,
    defenders_names=defenders_names,
    teammates_names=teammates_names,
    defenders_detected=defenders_detected,
    goalkeeper_xy=goalkeeper_xy,
    title_suffix=f" | Frame {frame_record['frame']} ({frame_record['timestamp']})",
    save_path=output_figure_path,
)
plt.show()

In [ ]:
# ==============================================================================
# Cell 6: Temporal Evolution of Passing Lanes Across Build-Up Phase (25fps)
# ==============================================================================
temporal_records = []

# Evaluate every 2 frames across the first 100 frames (4.0s build-up & wing progression)
for idx in range(0, 100, 2):
    fr = frames[idx]
    t_sec = idx / 25.0
    vels_t = compute_frame_velocities(frames, frame_idx=idx, window=5, fps=25.0)
    b_xy = np.array([fr["ball_data"]["x"], fr["ball_data"]["y"]], dtype=np.float64)

    p_coords = {p["player_id"]: np.array([p["x"], p["y"]], dtype=np.float64) for p in fr["player_data"]}
    defs_t = [
        p for p in fr["player_data"]
        if p["player_id"] in active_pids
        and p["player_id"] != gk_id
        and "Perth Glory" in team_lookup.get(p["player_id"], {}).get("team_name", "")
    ]
    defs_xy_t = np.array([[d["x"], d["y"]] for d in defs_t], dtype=np.float64)
    defs_vel_t = np.array([vels_t.get(d["player_id"], np.zeros(2)) for d in defs_t], dtype=np.float64)

    # Dynamic targets moving with teammates
    t_brazete = p_coords[965698]
    t_wing_run = np.array([min(48.0, t_brazete[0] + 4.5), min(31.0, t_brazete[1] + 6.5)])
    t_central = np.array([25.0, 0.0])
    t_jelacic = p_coords[560986]

    temporal_records.append({
        "t_sec": t_sec,
        "Wide Wing Run (3D Pose)": engine.compute_frame_occlusion(b_xy, t_wing_run, defs_xy_t, defs_vel_t, defs_t, use_pose=True),
        "Wide Wing Run (2D Baseline)": engine.compute_frame_occlusion(b_xy, t_wing_run, defs_xy_t, defs_vel_t, defs_t, use_pose=False),
        "Right Flank Feet (J. Brazete)": engine.compute_frame_occlusion(b_xy, t_brazete, defs_xy_t, defs_vel_t, defs_t, use_pose=True),
        "Central Line Break": engine.compute_frame_occlusion(b_xy, t_central, defs_xy_t, defs_vel_t, defs_t, use_pose=True),
        "Left Half-Space (K. Jelacic)": engine.compute_frame_occlusion(b_xy, t_jelacic, defs_xy_t, defs_vel_t, defs_t, use_pose=True),
    })

df_temporal = pd.DataFrame(temporal_records)

fig_temp, ax_t = plt.subplots(figsize=(11, 4.5), dpi=200)
ax_t.plot(df_temporal["t_sec"], df_temporal["Central Line Break"], color="#d62728", lw=2.2, label="Central Line Break (Blocked by Pennington)")
ax_t.plot(df_temporal["t_sec"], df_temporal["Right Flank Feet (J. Brazete)"], color="#ff7f0e", lw=2.0, label="Right Flank to Feet (J. Brazete)")
ax_t.plot(df_temporal["t_sec"], df_temporal["Left Half-Space (K. Jelacic)"], color="#9467bd", lw=1.8, ls="-.", label="Left Half-Space (K. Jelacic)")
ax_t.plot(df_temporal["t_sec"], df_temporal["Wide Wing Run (3D Pose)"], color="#2ca02c", lw=2.6, label="Wide Wing Space (DLOI-Pose 3D)")
ax_t.plot(df_temporal["t_sec"], df_temporal["Wide Wing Run (2D Baseline)"], color="#2ca02c", lw=1.4, ls="--", alpha=0.65, label="Wide Wing Space (2D Isotropic Baseline)")

ax_t.axvline(2.40, color="#111111", ls=":", lw=1.5, label="Actual Pass Release (t = 2.40s, Frame 124547)")
ax_t.set_title("Temporal Evolution of Passing Lane Occlusion During Build-Up (Phase 406)", fontsize=11, weight="bold")
ax_t.set_xlabel("Phase Time [seconds] (25fps SkillCorner Stream)")
ax_t.set_ylabel("DLOI Occlusion Score [0 = Open, 1 = Blocked]")
ax_t.set_ylim(-0.02, 1.05)
ax_t.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=8.0, frameon=True)
ax_t.grid(True, ls=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Cell 7: Leave-One-Out Cover Shadow Attribution + 3D Scanning + Season OTIP Fusion
# ==============================================================================
loo_rows = []
scan_rows = []

for idx in range(0, 150, 5):
    fr = frames[idx]
    vels_i = compute_frame_velocities(frames, frame_idx=idx, window=5, fps=25.0)
    b_xy = np.array([fr["ball_data"]["x"], fr["ball_data"]["y"]], dtype=np.float64)
    p_map = {p["player_id"]: np.array([p["x"], p["y"]], dtype=np.float64) for p in fr["player_data"]}

    eval_targets = [
        np.array([25.0, 0.0]),
        p_map[965698],
        p_map[51002],
        p_map[560986],
        np.array([min(48.0, p_map[965698][0] + 4.5), min(31.0, p_map[965698][1] + 6.5)]),
    ]

    defs_i = [
        p for p in fr["player_data"]
        if p["player_id"] in active_pids
        and p["player_id"] != gk_id
        and "Perth Glory" in team_lookup.get(p["player_id"], {}).get("team_name", "")
    ]
    defs_xy_i = np.array([[d["x"], d["y"]] for d in defs_i], dtype=np.float64)
    defs_vel_i = np.array([vels_i.get(d["player_id"], np.zeros(2)) for d in defs_i], dtype=np.float64)

    full_dloi = [
        engine.compute_frame_occlusion(b_xy, tgt, defs_xy_i, defs_vel_i, defs_i, use_pose=True)
        for tgt in eval_targets
    ]

    for j, d in enumerate(defs_i):
        pid = d["player_id"]
        mask = [k for k in range(len(defs_i)) if k != j]
        without_dloi = [
            engine.compute_frame_occlusion(
                b_xy, tgt, defs_xy_i[mask], defs_vel_i[mask], [defs_i[k] for k in mask], use_pose=True
            )
            for tgt in eval_targets
        ]
        deltas = [max(0.0, f_val - w_val) for f_val, w_val in zip(full_dloi, without_dloi)]
        loo_rows.append({
            "player_id": pid,
            "Player": team_lookup[pid]["short_name"],
            "Role": team_lookup[pid]["position_group"],
            "peak_shadow": max(deltas),
            "mean_shadow": np.mean(deltas),
        })

# Compute 3D Head-Torso scanning angle across all 309 frames
for idx, fr in enumerate(frames):
    vels_i = compute_frame_velocities(frames, frame_idx=idx, window=5, fps=25.0)
    for p in fr["player_data"]:
        pid = p["player_id"]
        if pid in active_pids and pid != gk_id and "Perth Glory" in team_lookup.get(pid, {}).get("team_name", ""):
            vt = engine.extract_torso_vector(p)
            vg = engine.extract_gaze_vector(p)
            if vt is not None and vg is not None:
                ang = math.degrees(math.acos(float(np.clip(np.dot(vt, vg), -1.0, 1.0))))
                scan_rows.append({
                    "player_id": pid,
                    "scan_deg": ang,
                    "speed": float(np.linalg.norm(vels_i.get(pid, np.zeros(2)))),
                })

df_loo_sum = pd.DataFrame(loo_rows).groupby(["player_id", "Player", "Role"], as_index=False).agg(
    Peak_Shadow_Added=("peak_shadow", lambda s: round(float(np.max(s)), 3)),
    Mean_Shadow_Added=("mean_shadow", lambda s: round(float(np.mean(s)), 3)),
)
df_scan_sum = pd.DataFrame(scan_rows).groupby("player_id", as_index=False).agg(
    Mean_Scan_Deg=("scan_deg", lambda s: round(float(np.mean(s)), 1)),
    P90_Scan_Deg=("scan_deg", lambda s: round(float(np.percentile(s, 90)), 1)),
    Phase_Speed_m_s=("speed", lambda s: round(float(np.mean(s)), 2)),
)

df_audit = pd.merge(df_loo_sum, df_scan_sum, on="player_id", how="left")

# Fuse with season-level SkillCorner Physical Aggregates (OTIP - Out of Possession)
phy_paths = list(project_root.rglob("aus1league_physicalaggregates_20242025.csv"))
if phy_paths:
    phy_df = pd.read_csv(phy_paths[0])
    phy_dedup = phy_df.groupby("player_id", as_index=False).agg(
        Season_OTIP_m_min=("total_metersperminute_full_otip", lambda s: round(float(np.mean(s)), 1)),
        Season_PSV99_kmh=("psv99", lambda s: round(float(np.max(s)), 1)),
    )
    df_audit = pd.merge(df_audit, phy_dedup, on="player_id", how="left")

df_audit = df_audit.drop(columns=["player_id"]).sort_values(by="Mean_Shadow_Added", ascending=False).reset_index(drop=True)
display(df_audit)